# Menstrual health, sleep and wrist temperature

Academic exploration of sleep, hormone measurements and wrist temperature using mcPHASES version 1.0.0. Four input files are required; see [README](README.md). Raw and derived participant data and saved outputs are excluded under the [PhysioNet Restricted Health Data Licence](https://physionet.org/content/mcphases/view-license/1.0.0/).


## Dataset overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import seaborn as sns

In [ ]:
# Load the tables used in the sleep analysis.
# Wrist-temperature data is loaded later, in its own section.
files = {
    "hormones": "hormones_and_selfreport.csv",
    "sleep": "sleep.csv",
    "sleep_score": "sleep_score.csv",
}
datasets = {name: pd.read_csv(file, low_memory=False) for name, file in files.items()}
for name, table in datasets.items():
    print(f"{name}: {table.shape}")


In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df.columns)

In [ ]:
for name, df in datasets.items():
    print("\n======================")
    print(name)
    print("======================")

    print("\nmissing values:")
    print((df.isna().mean()*100).sort_values(ascending=False).head(10))

    print("\nduplicates:", df.duplicated().sum())

In [ ]:
for name, df in datasets.items():

    numeric_df = df.select_dtypes(include=["number"])

    if numeric_df.shape[1] > 0:
        print("\n======================")
        print(name)
        print("======================")

        display(numeric_df.describe().T)

In [ ]:
for name, df in datasets.items():
    print("\n" + "="*70)
    print(f"DATASET: {name}")
    print("="*70)
    print("\nFirst 5 rows:")
    display(df.head())

In [ ]:
# Sleep EDA again 

## Sleep and hormone analysis

In [ ]:
horm = datasets["hormones"].copy()
sleep_score = datasets["sleep_score"].copy()
sleep = datasets["sleep"].copy()
print("Loaded shapes:", horm.shape, sleep_score.shape, sleep.shape)


In [ ]:
print("Hormone/self-report columns:")
print(horm.columns)

print("\nSleep score columns:")
print(sleep_score.columns)

print("\nSleep columns:")
print(sleep.columns)

In [ ]:
horm.head()
sleep_score.head()
sleep.head()

In [ ]:
# cleaning data

horm["phase"] = horm["phase"].astype("string").str.strip()

# Numeric conversion for hormone columns
for col in ["lh", "estrogen", "pdg"]:
    horm[col] = pd.to_numeric(horm[col], errors="coerce")

# Numeric conversion for sleep_score columns
sleep_score_numeric = [
    "overall_score",
    "composition_score",
    "revitalization_score",
    "duration_score",
    "deep_sleep_in_minutes",
    "resting_heart_rate",
    "restlessness"
]
for col in sleep_score_numeric:
    sleep_score[col] = pd.to_numeric(sleep_score[col], errors="coerce")

# Numeric conversion for sleep columns
sleep_numeric = [
    "duration",
    "minutestofallasleep",
    "minutesasleep",
    "minutesawake",
    "minutesafterwakeup",
    "timeinbed",
    "efficiency"
]
for col in sleep_numeric:
    sleep[col] = pd.to_numeric(sleep[col], errors="coerce")


print("Numeric conversion done.")

In [ ]:
# Keep only main sleep events for the session-level sleep table
sleep_main = sleep[sleep["mainsleep"] == True].copy()

In [ ]:
# merging data

# Main analysis merge: hormones/self-report + sleep_score
df = horm.merge(
    sleep_score,
    on=["id", "study_interval", "is_weekend", "day_in_study"],
    how="inner"
)

print("\nMerged hormone + sleep_score shape:", df.shape)
print("Unique participants in merged data:", df["id"].nunique())
print("Phase counts:")
print(df["phase"].value_counts(dropna=False))

# second merge with sleep session data
# We align sleep end day to scored day because sleep_score is recorded for the morning/end of sleep
sleep_main_for_merge = sleep_main.rename(columns={"sleep_end_day_in_study": "day_in_study"}).copy()

df_full = df.merge(
    sleep_main_for_merge[
        ["id", "study_interval", "is_weekend", "day_in_study",
         "minutesasleep", "minutesawake", "timeinbed", "efficiency",
         "sleep_start_timestamp", "sleep_end_timestamp", "type"]
    ],
    on=["id", "study_interval", "is_weekend", "day_in_study"],
    how="left"
)

print("\nMerged full shape:", df_full.shape)
display(df_full.head())

# hours slept
df_full["sleep_hours"] = df_full["minutesasleep"] / 60

In [ ]:


print("\n===== MISSINGNESS (%) =====")
missing_pct = (df_full.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.round(2))

print("\n===== DESCRIPTIVE STATS =====")
desc_cols = [
    "overall_score", "composition_score", "revitalization_score", "duration_score",
    "deep_sleep_in_minutes", "resting_heart_rate", "restlessness",
    "minutesasleep", "sleep_hours", "efficiency",
    "lh", "estrogen", "pdg"
]
print(df_full[desc_cols].describe().T)

print("\n===== DUPLICATES =====")
print("Exact duplicate rows in merged full dataset:", df_full.duplicated().sum())

print("\n===== OBSERVATIONS PER PHASE =====") #how many rows belong to each phase
print(df_full["phase"].value_counts())

# Save a clean merged file for later use
df_full.to_csv("sleep_analysis_merged.csv", index=False)
print("\nSaved merged dataset to sleep_analysis_merged.csv")

In [ ]:
# functions for plotting

def save_hist(data, col, bins=30):
    plt.figure(figsize=(7, 5))
    plt.hist(data[col].dropna(), bins=bins)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

def save_boxplot(data, xcol, ycol, order=None):
    groups = []
    labels = []
    if order is None:
        order = sorted(data[xcol].dropna().unique())
    for cat in order:
        vals = data.loc[data[xcol] == cat, ycol].dropna()
        if len(vals) > 0:
            groups.append(vals)
            labels.append(cat)

    plt.figure(figsize=(8, 5))
    plt.boxplot(groups, tick_labels=labels)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.title(f"{ycol} by {xcol}")
    plt.tight_layout()
    plt.show()

def save_scatter(data, xcol, ycol):
    scatter = data[[xcol, ycol]].dropna()
    plt.figure(figsize=(6, 5))
    plt.scatter(scatter[xcol], scatter[ycol], alpha=0.5)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.title(f"{ycol} vs {xcol}")
    plt.tight_layout()
    plt.show()
    
def save_missing_bar(data, cols, filename="missingness_sleep_vars.png"):
    miss = data[cols].isna().mean().sort_values(ascending=False) * 100
    plt.figure(figsize=(9, 5))
    plt.bar(miss.index, miss.values)
    plt.ylabel("Missing values (%)")
    plt.title("Missingness in selected sleep variables")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()  
    
def save_corr_heatmap(data, cols, filename="sleep_hormone_corr_heatmap.png"):
    corr = data[cols].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, aspect="auto")
    ax.set_xticks(range(len(cols)))
    ax.set_yticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticklabels(cols)
    ax.set_title("Spearman correlation heatmap")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Correlation")
    plt.tight_layout()
    plt.show()


In [ ]:
phase_order = ["Menstrual", "Follicular", "Fertility", "Luteal"]

df_full["phase"] = pd.Categorical(
    df_full["phase"],
    categories=phase_order,
    ordered=True
)

In [ ]:
# visuals

# Missingness bar chart
sleep_vars_for_missing = [
    "overall_score", "composition_score", "revitalization_score", "duration_score",
    "deep_sleep_in_minutes", "resting_heart_rate", "restlessness",
    "minutesasleep", "sleep_hours", "efficiency"
]

miss = df_full[sleep_vars_for_missing].isna().mean().sort_values(ascending=False) * 100

plt.figure(figsize=(9, 5))
plt.bar(miss.index, miss.values)
plt.ylabel("Missing values (%)")
plt.title("Missingness in selected sleep variables")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show() 



In [ ]:
phase_sleep = df_full.groupby("phase", observed= False)["overall_score"].mean()

phase_sleep.plot(kind="bar")

plt.ylabel("Average Sleep Score")
plt.title("Average Sleep Score by Menstrual Phase")
plt.show()

In [ ]:

df_full.boxplot(
    column="sleep_hours",
    by="phase"
)

plt.title("Sleep Duration by Menstrual Phase")
plt.suptitle("")
plt.ylabel("Hours of Sleep")
plt.show()

In [ ]:


df_full.boxplot(column="overall_score", by="phase")

plt.title("Sleep Score Across Menstrual Phases")
plt.suptitle("")
plt.xlabel("Menstrual Phase")
plt.ylabel("Sleep Score")

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(df_full["overall_score"].dropna(), bins=25)
plt.xlabel("overall_score")
plt.ylabel("Count")
plt.title("Distribution of overall sleep score")
plt.tight_layout()
plt.show()

In [ ]:
#no strong estrogen-sleep relationship
plt.scatter(df_full["estrogen"], df_full["overall_score"], alpha=0.4)

plt.xlabel("Estrogen Level")
plt.ylabel("Sleep Score")
plt.title("Relationship between Estrogen and Sleep Score")

plt.show()

In [ ]:
plt.scatter(df_full["pdg"], df_full["overall_score"], alpha=0.4)

plt.xlabel("Pdg Level")
plt.ylabel("Sleep Score")
plt.title("Relationship between Pdg and Sleep Score")

plt.show()

In [ ]:
plt.scatter(df_full["lh"], df_full["overall_score"], alpha=0.4)

plt.xlabel("LHLevel")
plt.ylabel("Sleep Score")
plt.title("Relationship between LH and Sleep Score")

plt.show()

In [ ]:
corr_cols = [
    "overall_score", "deep_sleep_in_minutes", "resting_heart_rate",
    "restlessness", "sleep_hours", "efficiency",
    "lh", "estrogen", "pdg"
]

corr = df_full[corr_cols].corr(method="spearman")
plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar(label="Correlation")
plt.title("Spearman correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# 7. PHASE-WISE SUMMARY TABLE

phase_summary = df_full.groupby("phase", observed= False).agg(
    n_obs=("id", "size"),
    mean_sleep_score=("overall_score", "mean"),
    median_sleep_score=("overall_score", "median"),
    mean_deep_sleep=("deep_sleep_in_minutes", "mean"),
    mean_restlessness=("restlessness", "mean"),
    mean_rhr=("resting_heart_rate", "mean"),
    mean_sleep_hours=("sleep_hours", "mean"),
    mean_efficiency=("efficiency", "mean"),
).round(2)

print("\n===== PHASE-WISE SUMMARY =====")
print(phase_summary)


## Statistical comparisons

In [ ]:
#  STATISTICAL TESTS
def kruskal_by_phase(data, outcome, phase_col="phase", order=None):
    if order is None:
        order = sorted(data[phase_col].dropna().unique())
    groups = [data.loc[data[phase_col] == p, outcome].dropna() for p in order]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        return stat, p
    return np.nan, np.nan

test_vars = ["overall_score", "deep_sleep_in_minutes", "restlessness", "resting_heart_rate", "sleep_hours"]

results = []
for var in test_vars:
    stat, p = kruskal_by_phase(df_full, var, order=phase_order)
    results.append({"variable": var, "kruskal_H": stat, "p_value": p})

results_df = pd.DataFrame(results)
print("\n===== KRUSKAL-WALLIS TESTS =====")
print(results_df)

results_df.to_csv("kruskal_results_sleep.csv", index=False)
print("\nSaved kruskal_results_sleep.csv")

In [ ]:
# SPEARMAN CORRELATIONS WITH HORMONES

corr_results = []
for hormone in ["lh", "estrogen", "pdg"]:
    for sleep_var in ["overall_score", "deep_sleep_in_minutes", "restlessness", "resting_heart_rate", "sleep_hours"]:
        tmp = df_full[[hormone, sleep_var]].dropna()
        if len(tmp) > 2:
            rho, p = stats.spearmanr(tmp[hormone], tmp[sleep_var])
            corr_results.append({
                "hormone": hormone,
                "sleep_variable": sleep_var,
                "n": len(tmp),
                "spearman_rho": rho,
                "p_value": p
            })

corr_results_df = pd.DataFrame(corr_results)
print("\n===== SPEARMAN CORRELATIONS =====")
print(corr_results_df.sort_values("p_value").head(20))

corr_results_df.to_csv("spearman_sleep_hormones.csv", index=False)
print("\nSaved spearman_sleep_hormones.csv")


In [ ]:
# PARTICIPANT-LEVEL EXAMPLE PLOT (How sleep score changes over time for individual participants)

# This is nice for showing longitudinal variation, but not for all participants in one plot.

top_ids = df_full["id"].value_counts().head(4).index.tolist()

plt.figure(figsize=(9, 5))
for pid in top_ids:
    sub = df_full[df_full["id"] == pid].sort_values("day_in_study")
    plt.plot(sub["day_in_study"], sub["overall_score"], marker="o", label=f"id {pid}")

plt.xlabel("day_in_study")
plt.ylabel("overall_score")
plt.title("Example participant-level sleep score trajectories")
plt.legend()
plt.tight_layout()
plt.savefig( "participant_sleep_trajectories.png", dpi=300)
plt.show()

print("\nAll main figures saved as PNG files in /mnt/data/")

In [ ]:
df_full.boxplot(
    column="lh",
    by="phase"
)

plt.title("LH levels across menstrual phases")
plt.suptitle("")
plt.xlabel("Phase")
plt.ylabel("LH level")
plt.show()

In [ ]:
df_full.boxplot(
    column="pdg",
    by="phase"
)

plt.title("PdG levels across menstrual phases")
plt.suptitle("")
plt.xlabel("Phase")
plt.ylabel("PdG level")
plt.show()

In [ ]:
df_full.boxplot(
    column="estrogen",
    by="phase"
)

plt.title("Estrogen levels across menstrual phases")
plt.suptitle("")
plt.xlabel("Phase")
plt.ylabel("Estrogen level")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.violinplot(
    x="phase",
    y="overall_score",
    data=df_full,
    inner="quartile"
)

plt.title("Sleep score distribution across menstrual phases")
plt.show()

## Wrist temperature analysis

In [ ]:
wrist = pd.read_csv("wrist_temperature.csv")
horm = pd.read_csv("hormones_and_selfreport.csv")

print("Wrist shape:", wrist.shape)
print("Hormones shape:", horm.shape)
# Normalize the alternate temperature-column spelling.
if 'temp_diff_from_baseline' not in wrist.columns and 'temperature_diff_from_baseline' in wrist.columns:
    wrist = wrist.rename(columns={'temperature_diff_from_baseline':'temp_diff_from_baseline'})


In [ ]:
print("nWrist columns:")
print(wrist.columns)

In [ ]:

#  TEMP COLUMN
# -------------------------
temp_col = [c for c in wrist.columns if "temp" in c.lower()][0]
print("Using temperature column:", temp_col)



In [ ]:
# CLEAN WRIST DATA

wrist = wrist[["id", "day_in_study", temp_col]].copy()

wrist[temp_col] = pd.to_numeric(wrist[temp_col], errors="coerce")

print("Wrist duplicates:", wrist.duplicated().sum())
wrist = wrist.drop_duplicates()

wrist = wrist.dropna(subset=[temp_col])

print("Clean wrist shape:", wrist.shape)



In [ ]:
# -------------------------
# 4. CLEAN HORMONE DATA
# -------------------------
horm["phase"] = horm["phase"].astype("string").str.strip()

horm = horm[["id", "day_in_study", "phase"]].copy()

print("Horm duplicates:", horm.duplicated().sum())
horm = horm.drop_duplicates()

horm = horm.dropna(subset=["phase"])

print("Clean hormone shape:", horm.shape)



In [ ]:
# -------------------------
# 5. MERGE DATA
# -------------------------
df_temp = wrist.merge(
    horm,
    on=["id", "day_in_study"],
    how="inner"
)

print("Merged shape:", df_temp.shape)
print(df_temp.head())



In [ ]:
# -------------------------
# 6. PHASE ORDER
# -------------------------
phase_order = ["Menstrual", "Follicular", "Fertility", "Luteal"]

df_temp["phase"] = pd.Categorical(
    df_temp["phase"],
    categories=phase_order,
    ordered=True
)



In [ ]:
# -------------------------
# 7. EDA SUMMARIES
# -------------------------
print("\n===== MISSINGNESS (%) =====")
print((df_temp.isna().mean()*100).round(2))

print("\n===== DESCRIPTIVE STATS =====")
print(df_temp[[temp_col]].describe().T)

print("\n===== OBSERVATIONS PER PHASE =====")
print(df_temp["phase"].value_counts())



In [ ]:
# -------------------------
# 8. DISTRIBUTION
# -------------------------
plt.figure(figsize=(6,4))
plt.hist(df_temp[temp_col], bins=30)
plt.title("Distribution of wrist temperature (diff from baseline)")
plt.xlabel("Temperature difference")
plt.ylabel("Count")
plt.show()



In [ ]:
# -------------------------
# 9. BOXPLOT
# -------------------------
plt.figure(figsize=(7,5))
sns.boxplot(x="phase", y=temp_col, data=df_temp, order=phase_order)
plt.title("Wrist temperature (diff from baseline) across phases")
plt.show()



In [ ]:
# -------------------------
# 10. VIOLIN PLOT
# -------------------------
plt.figure(figsize=(7,5))
sns.violinplot(x="phase", y=temp_col, data=df_temp, order=phase_order)
plt.title("Distribution of wrist temperature across phases")
plt.show()



In [ ]:
# -------------------------
# 11. MEAN PLOT
# -------------------------
mean_temp = df_temp.groupby("phase", observed=False)[temp_col].mean().reindex(phase_order)

plt.figure(figsize=(7,5))
mean_temp.plot(kind="bar")
plt.title("Mean wrist temperature difference per phase")
plt.ylabel("Temperature difference")
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- ASSUMES YOU ALREADY HAVE df_temp ----------
# df_temp should contain: id, day_in_study, phase, temp_diff_from_baseline

temp_col = "temp_diff_from_baseline"

phase_order = ["Menstrual", "Follicular", "Fertility", "Luteal"]

# Clean and order phase
df_temp = df_temp.dropna(subset=["id", "day_in_study", "phase", temp_col]).copy()
df_temp["phase"] = df_temp["phase"].astype("string").str.strip()
df_temp["phase"] = pd.Categorical(df_temp["phase"], categories=phase_order, ordered=True)

# Sort by participant and time
df_temp = df_temp.sort_values(["id", "day_in_study"]).copy()

# Mark when phase changes within each participant
df_temp["prev_phase"] = df_temp.groupby("id")["phase"].shift(1)
df_temp["phase_change"] = (df_temp["phase"] != df_temp["prev_phase"]).astype(int)

# Create phase blocks
df_temp["phase_block"] = df_temp.groupby("id")["phase_change"].cumsum()

# Summarize each contiguous phase block
phase_blocks = (
    df_temp.groupby(["id", "phase", "phase_block"], observed=False)
    .agg(
        start_day=("day_in_study", "min"),
        end_day=("day_in_study", "max"),
        n_records=("day_in_study", "size")
    )
    .reset_index()
)

# Length in days
phase_blocks["length_days"] = phase_blocks["end_day"] - phase_blocks["start_day"] + 1

print("Sample phase blocks:")
print(phase_blocks.head())

# Summary by phase
phase_length_summary = (
    phase_blocks.groupby("phase", observed=False)["length_days"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
    .reindex(phase_order)
)

print("\nPhase length summary:")
print(phase_length_summary)